# Kaggle version: Original và MoE classification head

In [ ]:
from pathlib import Path
import os

# Dùng repo đã clone/upload; chọn đường dẫn phù hợp phiên làm việc.
REPO_DIR = Path("/content/code") if "COLAB_RELEASE_TAG" in os.environ else Path("/kaggle/working/code")

if Path("train.py").exists():
    REPO_DIR = Path.cwd()

REPOSITORY_URL = "https://github.com/lngphgthao/moe-vihsd.git"
BRANCH = "main" 

if not REPO_DIR.exists():
    !git clone --depth 1 --branch "$BRANCH" "$REPOSITORY_URL" "$REPO_DIR"
    
assert (REPO_DIR / "train.py").is_file(), "Đặt REPO_DIR đến checkout/upload chứa code mới"
os.chdir(REPO_DIR)
!python -m pip install -r requirements.txt
assert _exit_code == 0, "Cài dependency thất bại"

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print(f"Chưa lấy được secret {key}; thêm vào Kaggle Secrets nếu cần")
os.environ["RUNS_DIR"] = "/kaggle/working/runs"

In [ ]:
!python -m unittest discover -s tests -v
assert _exit_code == 0, "Test chưa đạt; dừng trước khi chạy experiment"

## Screening backbone original head

Session 1 chọn DATASET="vianli": exp-003/XLM-R và exp-007/mDeBERTa.
Session 2 chọn DATASET="vihsd": exp-008/XLM-R và exp-009/mDeBERTa.
Mỗi group chạy hai backbone tuần tự. ViHSD dùng text gốc từ Hub, cần HF_TOKEN có quyền truy cập.
Chỉ dùng run ID chưa chạy; giữ nguyên ID failed và đăng ký ID mới khi retry.
Smoke: run-010/011 (ViANLI), run-012/013 (ViHSD), tắt W&B.
Full: run-003/007 (ViANLI), run-008/009 (ViHSD), bật W&B theo config.


In [ ]:
DATASET = "vianli"  # Session 2 đổi thành "vihsd".
assert DATASET in ("vianli", "vihsd")
!python scripts/run_group.py --group-config configs/groups/{DATASET}_multilingual_smoke.yaml --smoke-test
assert _exit_code == 0, "Smoke thất bại; không chạy cell full"

## Full run sau khi smoke đạt

Chỉ chạy cell này khi cả hai smoke completed, loss hữu hạn và có checkpoint.
Full dùng 7 epoch, batch 16, seed 42; chọn checkpoint bằng validation macro F1.
Lệnh ! thất bại không tự ngăn cell tiếp theo; không Run All khi chưa kiểm tra smoke.


In [ ]:
!python scripts/run_group.py --group-config configs/groups/{DATASET}_multilingual_screening.yaml --no-smoke-test
assert _exit_code == 0, "Full run thất bại; xem log và giữ artifact"

In [ ]:
!python scripts/aggregate_results.py --runs-dir /kaggle/working/runs --output-dir /kaggle/working/runs/analysis
assert _exit_code == 0, "Tổng hợp kết quả thất bại"

!cp experiments/experiment_tracker.csv /kaggle/working/experiment_tracker.csv
assert _exit_code == 0, "Lưu tracker thất bại"

## Test sau khi khóa finalist

Chỉ chạy evaluate khi config và seed đã khóa; lệnh mẫu bên dưới để chủ động sửa/chạy.
Giữ toàn bộ run directory, tracker CSV và W&B URL trước khi kết thúc phiên GPU.


In [ ]:
# !python evaluate.py --run-dir /kaggle/working/runs/<run-id> --finalist-config configs/finalists/<dataset>/<experiment-id>.yaml --final-evaluation